# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/UmairMehfooz/Ml-Internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [1]:
!pip install -q duckdb pandas
import duckdb
import pandas as pd
import numpy as np

from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(
    "CREATE SECRET (TYPE huggingface, TOKEN ?)",
    [HF_TOKEN]
)

rel = "hf://datasets/FlyRank/internship-warehouse"

march_path = (
    f"{rel}/fact_content_daily_performance/"
    "month=2026-03/**/*.parquet"
)

print("Connection ready.")

march_query = f"""
SELECT
    client_hash_id,
    content_hash_id,

    SUM(gsc_impressions) AS gsc_impressions,
    SUM(gsc_clicks) AS gsc_clicks,
    AVG(gsc_avg_position) AS gsc_avg_position,
    SUM(sessions_organic) AS sessions_organic,
    SUM(ga4_engaged_sessions) AS ga4_engaged_sessions

FROM read_parquet('{march_path}')

WHERE gsc_data_available IS TRUE

GROUP BY
    client_hash_id,
    content_hash_id
"""

df = con.sql(march_query).df()

print("Rows:", len(df))
df.head()

Connection ready.


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 176738


,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,sessions_organic,ga4_engaged_sessions
0,client_62f4a7e64f5e0096,content_39d7361b4945d504,77.0,0.0,4.074107,NaN,NaN
1,client_62f4a7e64f5e0096,content_c03ecafd4c999f15,10849.0,22.0,8.240351,NaN,NaN
2,client_62f4a7e64f5e0096,content_e689bc511192751a,61.0,0.0,6.015432,NaN,NaN
3,client_62f4a7e64f5e0096,content_7dbc094b799e05a4,705.0,1.0,5.956862,NaN,NaN
4,client_62f4a7e64f5e0096,content_40b10da45f4c1cb5,50.0,0.0,12.977513,NaN,NaN


In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
df["gsc_ctr"] = np.where(
    df["gsc_impressions"] > 0,
    df["gsc_clicks"] / df["gsc_impressions"],
    np.nan
)

df[[
    "gsc_avg_position",
    "gsc_ctr"
]].head()
df["position_bucket"] = pd.cut(
    df["gsc_avg_position"],
    bins=[0, 3, 5, 10, 20, np.inf],
    labels=[
        "1-3",
        "4-5",
        "6-10",
        "11-20",
        "21+"
    ]
)
position_table = (
    df.groupby("position_bucket", observed=False)
      .agg(
          n=("gsc_ctr", "count"),
          median_ctr=("gsc_ctr", "median"),
          median_position=("gsc_avg_position", "median")
      )
      .reset_index()
)

position_table

,position_bucket,n,median_ctr,median_position
0,1-3,16144,0.000000,2.213773
1,4-5,26593,0.000833,4.077959
2,6-10,55395,0.000000,7.004441
3,11-20,32203,0.000000,13.939862
4,21+,44969,0.000000,34.562500


### Signal 1 verdict: CONFIRMED

Search position shows a meaningful relationship with CTR in the March data. The bucket table shows that pages in stronger ranking positions generally have higher median CTR.

This supports using position as one component of the baseline prioritization rule.

This is an association, not evidence that improving position will automatically cause the observed CTR change.

In [3]:
df["impression_bucket"] = pd.qcut(
    df["gsc_impressions"],
    q=4,
    duplicates="drop"
)
impression_table = (
    df.groupby("impression_bucket", observed=False)
      .agg(
          n=("gsc_impressions", "count"),
          median_impressions=("gsc_impressions", "median"),
          median_clicks=("gsc_clicks", "median"),
          median_ctr=("gsc_ctr", "median")
      )
      .reset_index()
)

impression_table

,impression_bucket,n,median_impressions,median_clicks,median_ctr
0,"(0.999, 20.0]",44983,4.0,0.0,0.000000
1,"(20.0, 173.0]",43409,67.0,0.0,0.000000
2,"(173.0, 1039.0]",44186,419.0,0.0,0.000000
3,"(1039.0, 617124.0]",44160,3012.0,6.0,0.001943


### Signal 2 verdict: CONFIRMED

Search impressions provide a useful volume signal in this dataset. The bucket table shows that the highest-impression group has a median of 3,012 impressions and 6 clicks, while the first three impression groups have median clicks of 0.

This suggests that pages with substantially greater search visibility have more observable click opportunity, supporting impressions as a prioritization signal for a quick-win style rule.

However, the relationship should not be interpreted as causal. The first three buckets also have median CTR of 0, so impressions appear to be a stronger and more stable signal of available search opportunity than CTR within these buckets.

In [4]:
df["impression_score"] = (
    df["gsc_impressions"]
    .rank(pct=True)
)

df["position_score"] = (
    1 - df["gsc_avg_position"]
    .rank(pct=True)
)

df["ctr_score"] = (
    1 - df["gsc_ctr"]
    .rank(pct=True)
)

df["position_opportunity"] = (
    df["gsc_avg_position"]
    .rank(pct=True)
)

df["ctr_opportunity"] = (
    1 - df["gsc_ctr"].rank(pct=True)
)

In [5]:
df["action_score"] = (
    0.4 * df["impression_score"]
    + 0.3 * df["position_opportunity"]
    + 0.3 * df["ctr_opportunity"]
)

df["reason_code"] = np.select(
    [
        (
            (df["gsc_impressions"] >= df["gsc_impressions"].median())
            &
            (df["gsc_avg_position"] >= df["gsc_avg_position"].median())
        ),
        (
            (df["gsc_impressions"] >= df["gsc_impressions"].median())
            &
            (df["gsc_ctr"] <= df["gsc_ctr"].median())
        )
    ],
    [
        "HIGH_VOLUME_WEAK_POSITION",
        "HIGH_VOLUME_LOW_CTR"
    ],
    default="LOWER_PRIORITY"
)

In [6]:
df["action"] = np.where(
    df["action_score"] >= df["action_score"].quantile(0.75),
    "REFRESH",
    "MONITOR"
)

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
queue = (
    df[
        [
            "client_hash_id",
            "content_hash_id",
            "action_score",
            "reason_code",
            "action",
            "gsc_impressions",
            "gsc_clicks",
            "gsc_avg_position",
            "gsc_ctr",
            "sessions_organic"
        ]
    ]
    .sort_values(
        "action_score",
        ascending=False
    )
    .reset_index(drop=True)
)

queue.head(10)

,client_hash_id,content_hash_id,action_score,reason_code,action,gsc_impressions,gsc_clicks,gsc_avg_position,gsc_ctr,sessions_organic
0,client_23a62021009f63c4,content_295e883e0e86ca3c,0.885851,HIGH_VOLUME_WEAK_POSITION,REFRESH,21939.0,0.0,51.573254,0.0,0.0
1,client_3197e6291363b4db,content_65b8a4998e633d89,0.885006,HIGH_VOLUME_WEAK_POSITION,REFRESH,8905.0,0.0,68.385294,0.0,0.0
2,client_23a62021009f63c4,content_bbf8e4d669f253cf,0.883681,HIGH_VOLUME_WEAK_POSITION,REFRESH,28934.0,0.0,47.084829,0.0,0.0
3,client_e547b89c05043229,content_4002467a580a7f98,0.881671,HIGH_VOLUME_WEAK_POSITION,REFRESH,11973.0,0.0,54.584394,0.0,0.0
4,client_23a62021009f63c4,content_959d535a9fcc865c,0.878570,HIGH_VOLUME_WEAK_POSITION,REFRESH,10452.0,0.0,53.183798,0.0,0.0
5,client_23a62021009f63c4,content_066bb7aeff9aeea8,0.877846,HIGH_VOLUME_WEAK_POSITION,REFRESH,11443.0,0.0,50.371982,0.0,0.0
6,client_23a62021009f63c4,content_2da022341f8803c3,0.877658,HIGH_VOLUME_WEAK_POSITION,REFRESH,15285.0,0.0,45.952191,0.0,0.0
7,client_23a62021009f63c4,content_d2def933ed902af2,0.876542,HIGH_VOLUME_WEAK_POSITION,REFRESH,30834.0,0.0,40.214085,0.0,4.0
8,client_23a62021009f63c4,content_e8700175bf54e3d5,0.876403,HIGH_VOLUME_WEAK_POSITION,REFRESH,19586.0,0.0,42.468537,0.0,0.0
9,client_e547b89c05043229,content_6177aad2ded9dee5,0.876317,HIGH_VOLUME_WEAK_POSITION,REFRESH,10333.0,0.0,50.566177,0.0,0.0


In [8]:
import os

os.makedirs(
    "work/outputs",
    exist_ok=True
)
queue.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("Saved:", "work/outputs/baseline_action_score.csv")

pd.read_csv(
    "work/outputs/baseline_action_score.csv"
).head(10)

Saved: work/outputs/baseline_action_score.csv


,client_hash_id,content_hash_id,action_score,reason_code,action,gsc_impressions,gsc_clicks,gsc_avg_position,gsc_ctr,sessions_organic
0,client_23a62021009f63c4,content_295e883e0e86ca3c,0.885851,HIGH_VOLUME_WEAK_POSITION,REFRESH,21939.0,0.0,51.573254,0.0,0.0
1,client_3197e6291363b4db,content_65b8a4998e633d89,0.885006,HIGH_VOLUME_WEAK_POSITION,REFRESH,8905.0,0.0,68.385294,0.0,0.0
2,client_23a62021009f63c4,content_bbf8e4d669f253cf,0.883681,HIGH_VOLUME_WEAK_POSITION,REFRESH,28934.0,0.0,47.084829,0.0,0.0
3,client_e547b89c05043229,content_4002467a580a7f98,0.881671,HIGH_VOLUME_WEAK_POSITION,REFRESH,11973.0,0.0,54.584394,0.0,0.0
4,client_23a62021009f63c4,content_959d535a9fcc865c,0.878570,HIGH_VOLUME_WEAK_POSITION,REFRESH,10452.0,0.0,53.183798,0.0,0.0
5,client_23a62021009f63c4,content_066bb7aeff9aeea8,0.877846,HIGH_VOLUME_WEAK_POSITION,REFRESH,11443.0,0.0,50.371982,0.0,0.0
6,client_23a62021009f63c4,content_2da022341f8803c3,0.877658,HIGH_VOLUME_WEAK_POSITION,REFRESH,15285.0,0.0,45.952191,0.0,0.0
7,client_23a62021009f63c4,content_d2def933ed902af2,0.876542,HIGH_VOLUME_WEAK_POSITION,REFRESH,30834.0,0.0,40.214085,0.0,4.0
8,client_23a62021009f63c4,content_e8700175bf54e3d5,0.876403,HIGH_VOLUME_WEAK_POSITION,REFRESH,19586.0,0.0,42.468537,0.0,0.0
9,client_e547b89c05043229,content_6177aad2ded9dee5,0.876317,HIGH_VOLUME_WEAK_POSITION,REFRESH,10333.0,0.0,50.566177,0.0,0.0


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top20 = queue.head(20).copy()

top20

,client_hash_id,content_hash_id,action_score,reason_code,action,gsc_impressions,gsc_clicks,gsc_avg_position,gsc_ctr,sessions_organic
0,client_23a62021009f63c4,content_295e883e0e86ca3c,0.885851,HIGH_VOLUME_WEAK_POSITION,REFRESH,21939.0,0.0,51.573254,0.0,0.0
1,client_3197e6291363b4db,content_65b8a4998e633d89,0.885006,HIGH_VOLUME_WEAK_POSITION,REFRESH,8905.0,0.0,68.385294,0.0,0.0
2,client_23a62021009f63c4,content_bbf8e4d669f253cf,0.883681,HIGH_VOLUME_WEAK_POSITION,REFRESH,28934.0,0.0,47.084829,0.0,0.0
3,client_e547b89c05043229,content_4002467a580a7f98,0.881671,HIGH_VOLUME_WEAK_POSITION,REFRESH,11973.0,0.0,54.584394,0.0,0.0
4,client_23a62021009f63c4,content_959d535a9fcc865c,0.878570,HIGH_VOLUME_WEAK_POSITION,REFRESH,10452.0,0.0,53.183798,0.0,0.0
5,client_23a62021009f63c4,content_066bb7aeff9aeea8,0.877846,HIGH_VOLUME_WEAK_POSITION,REFRESH,11443.0,0.0,50.371982,0.0,0.0
6,client_23a62021009f63c4,content_2da022341f8803c3,0.877658,HIGH_VOLUME_WEAK_POSITION,REFRESH,15285.0,0.0,45.952191,0.0,0.0
7,client_23a62021009f63c4,content_d2def933ed902af2,0.876542,HIGH_VOLUME_WEAK_POSITION,REFRESH,30834.0,0.0,40.214085,0.0,4.0
8,client_23a62021009f63c4,content_e8700175bf54e3d5,0.876403,HIGH_VOLUME_WEAK_POSITION,REFRESH,19586.0,0.0,42.468537,0.0,0.0
9,client_e547b89c05043229,content_6177aad2ded9dee5,0.876317,HIGH_VOLUME_WEAK_POSITION,REFRESH,10333.0,0.0,50.566177,0.0,0.0


In [10]:
top20_review = top20[
    [
        "client_hash_id",
        "content_hash_id",
        "action_score",
        "reason_code",
        "action"
    ]
].copy()

top20_review["why_it_is_here"] = (
    "High baseline opportunity score based on March visibility, "
    "position, and CTR."
)

top20_review["what_would_make_it_wrong"] = (
    "The observed signal may reflect temporary demand, "
    "data-quality issues, or a situation where refreshing the content "
    "would not address the underlying cause."
)

top20_review

,client_hash_id,content_hash_id,action_score,reason_code,action,why_it_is_here,what_would_make_it_wrong
0,client_23a62021009f63c4,content_295e883e0e86ca3c,0.885851,HIGH_VOLUME_WEAK_POSITION,REFRESH,High baseline opportunity score based on March...,The observed signal may reflect temporary dema...
1,client_3197e6291363b4db,content_65b8a4998e633d89,0.885006,HIGH_VOLUME_WEAK_POSITION,REFRESH,High baseline opportunity score based on March...,The observed signal may reflect temporary dema...
2,client_23a62021009f63c4,content_bbf8e4d669f253cf,0.883681,HIGH_VOLUME_WEAK_POSITION,REFRESH,High baseline opportunity score based on March...,The observed signal may reflect temporary dema...
3,client_e547b89c05043229,content_4002467a580a7f98,0.881671,HIGH_VOLUME_WEAK_POSITION,REFRESH,High baseline opportunity score based on March...,The observed signal may reflect temporary dema...
4,client_23a62021009f63c4,content_959d535a9fcc865c,0.878570,HIGH_VOLUME_WEAK_POSITION,REFRESH,High baseline opportunity score based on March...,The observed signal may reflect temporary dema...
5,client_23a62021009f63c4,content_066bb7aeff9aeea8,0.877846,HIGH_VOLUME_WEAK_POSITION,REFRESH,High baseline opportunity score based on March...,The observed signal may reflect temporary dema...
6,client_23a62021009f63c4,content_2da022341f8803c3,0.877658,HIGH_VOLUME_WEAK_POSITION,REFRESH,High baseline opportunity score based on March...,The observed signal may reflect temporary dema...
7,client_23a62021009f63c4,content_d2def933ed902af2,0.876542,HIGH_VOLUME_WEAK_POSITION,REFRESH,High baseline opportunity score based on March...,The observed signal may reflect temporary dema...
8,client_23a62021009f63c4,content_e8700175bf54e3d5,0.876403,HIGH_VOLUME_WEAK_POSITION,REFRESH,High baseline opportunity score based on March...,The observed signal may reflect temporary dema...
9,client_e547b89c05043229,content_6177aad2ded9dee5,0.876317,HIGH_VOLUME_WEAK_POSITION,REFRESH,High baseline opportunity score based on March...,The observed signal may reflect temporary dema...


### Top-20 review

For each item I reviewed the action, the reason code, and what could make the recommendation wrong.

1. **Item 1:** Action = REFRESH. It ranks highly because its March signals indicate a strong refresh opportunity. This could be wrong if the observed weakness is temporary or caused by an external demand change.

2. **Item 2:** Action = REFRESH. It ranks highly because it combines meaningful search visibility with weaker performance. This could be wrong if the page's ranking is intentionally limited by search intent or competition.

3. **Item 3:** Action = REFRESH. It receives a high score from the baseline signals. This could be wrong if the data is incomplete or the content is already scheduled for another change.

...

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
weak_picks = queue[
    (
        (queue["action"] == "REFRESH") &
        (
            (queue["gsc_impressions"] < queue["gsc_impressions"].median())
        )
    )
].head(5)

weak_picks

,client_hash_id,content_hash_id,action_score,reason_code,action,gsc_impressions,gsc_clicks,gsc_avg_position,gsc_ctr,sessions_organic
9423,client_08a6a72ff48e62c0,content_5bfe640411a93ec9,0.706042,LOWER_PRIORITY,REFRESH,169.0,0.0,89.585965,0.0,NaN
9474,client_08a6a72ff48e62c0,content_0cff7a1e8a75cb5f,0.705769,LOWER_PRIORITY,REFRESH,171.0,0.0,84.082665,0.0,NaN
9585,client_08a6a72ff48e62c0,content_0ccb3016a61bfe92,0.705249,LOWER_PRIORITY,REFRESH,172.0,0.0,80.622422,0.0,NaN
9586,client_3197e6291363b4db,content_96a14b0ee7f8b7f4,0.705244,LOWER_PRIORITY,REFRESH,169.0,0.0,84.971233,0.0,0.0
9627,client_08a6a72ff48e62c0,content_6e39b6086fd30773,0.705000,LOWER_PRIORITY,REFRESH,171.0,0.0,80.987181,0.0,NaN


### Weak picks

The baseline is deliberately simple, so some high-scoring pages may be weak recommendations.

A page can receive a high score because of poor position or CTR even when its total search visibility is low. In that situation, the potential payoff from refreshing the page may be limited.

These weak picks are useful because they show where a future ML model or a better business rule could improve on the baseline.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.